# 🏥 Tech Challenge — FIAP Pós 9IADT 2026 — Fase 1
## Sistema inteligente de apoio à triagem de mulheres vítimas de violência
---
> **Dados de treino**: Notificações históricas SINAN - ano de 2025 (`violencia_BR_25.csv`)  
> **Saída**: Probabilidade de tipo de violência e encaminhamento/exame ser necessário  
---

### 🎯 Modelos comparados neste notebook

| # | Modelo | Tipo | Justificativa |
|---|--------|------|---------------|
| 1 | **Random Forest** | Ensemble (Bagging) | robusto e estável |
| 2 | **XGBoost** | Ensemble (Boosting) | Alta performance em dados tabulares desbalanceados |
| 3 | **Logistic Regression** | Linear | Interpretável, rápido, boa base de comparação |

---

### ⚠️ Aviso Ético
> O sistema é um **apoio à decisão clínica**.
> A avaliação final é sempre responsabilidade do profissional de saúde.  
> Todos os dados tratados conforme **LGPD (Lei nº 13.709/2018)**.

---
## 1️⃣ Importar o dataset
---

In [1]:
import numpy as np
import pandas as pd
import os
import time
import zipfile

start = time.time()

# O dataset está comprimido em zip; extraímos na primeira execução
zip_path  = 'data/violencia_BR_25.zip'
csv_path  = 'data/violencia_BR_25.csv'

if not os.path.exists(csv_path):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('data/')
    print("✅ Arquivo CSV extraído do zip.")

dados = pd.read_csv(csv_path, sep=",", encoding="latin-1", low_memory=False)

print(f"⏱️  Tempo de carregamento: {time.time() - start:.2f}s")
print(f"📊 Shape total: {dados.shape}")


⏱️ Tempo de carregamento: 19.61s
📊 Shape total: (688329, 160)


In [ ]:
# Visão geral do dataset
print("=== Tipos de dados ===")
print(dados.dtypes.value_counts())
print()
print("=== Primeiros registros ===")
dados.head(3)


---

**Copiar dados de vítimas do sexo feminino**

---

In [3]:
# Filtrar apenas vítimas do sexo feminino (conforme objetivo do sistema)
dadosFemininos = dados[dados['CS_SEXO'] == 'F'].copy()

print(f"✅ Registros femininos: {dadosFemininos.shape[0]:,} linhas × {dadosFemininos.shape[1]} colunas")
print(f"\n📋 Amostra dos dados:")

dadosFemininos.head(5)

✅ Registros femininos: 477,994 linhas × 160 colunas

📋 Amostra dos dados:


,TP_NOT,ID_AGRAVO,DT_NOTIFIC,SEM_NOT,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_UNIDADE,DT_OCOR,SEM_PRI,...,CONS_IDO,DELEG_IDOS,DIR_HUMAN,MPU,DELEG_CRIA,DELEG_MULH,DELEG,INFAN_JUV,DEFEN_PUBL,DT_ENCERRA
0,2,Y09,2025-01-01,202501,2025,43,432160,2793008,2025-01-01,202501,...,2.0,2.0,2.0,2.0,2.0,1.0,2.0,2.0,2.0,2025-01-01
1,2,Y09,2025-01-01,202501,2025,33,330580,2297795,2025-01-01,202501,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2025-01-07
4,2,Y09,2025-01-01,202501,2025,33,330330,0012521,2025-01-01,202501,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2025-10-01
5,2,Y09,2025-01-01,202501,2025,33,330330,5935377,2025-01-01,202501,...,2.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,2.0,2025-01-01
7,2,Y09,2025-01-01,202501,2025,35,355030,6144055,2025-01-01,202501,...,2.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,2.0,2025-01-01


In [ ]:
# Informações gerais do subconjunto feminino
print(f"📋 Shape (feminino): {dadosFemininos.shape}")
print()
print("=== Colunas e tipos ===")
dadosFemininos.dtypes


---
## 2️⃣ Análise Exploratória de Dados (EDA)
---
> Carregamos a base, exploramos características gerais, estatísticas descritivas,
> distribuições e padrões específicos relacionados à saúde e segurança feminina.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

# Análise de valores ausentes / ignorados no subconjunto feminino
missing = (dadosFemininos.isnull().sum() / len(dadosFemininos) * 100)
missing = missing[missing > 0].sort_values(ascending=False).head(30)

plt.figure(figsize=(12, 6))
missing.plot(kind='bar', color='steelblue')
plt.title("Top 30 colunas com maior % de valores ausentes (registros femininos)")
plt.ylabel("% valores ausentes")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"Colunas sem nenhum valor ausente: {(dadosFemininos.isnull().sum() == 0).sum()}")
print(f"Colunas com > 80 % ausentes: {(dadosFemininos.isnull().mean() > 0.8).sum()}")


In [ ]:
# Extração da idade em anos a partir do campo codificado NU_IDADE_N
# Formato: primeiro dígito = unidade (4 = anos), demais dígitos = valor
def decode_age_years(val):
    if pd.isna(val):
        return np.nan
    v = int(val)
    unit = v // 1000   # 1=hora, 2=dia, 3=mês, 4=ano
    num  = v  % 1000
    if unit == 4:
        return num
    elif unit == 3:
        return num / 12
    elif unit == 2:
        return num / 365
    else:
        return 0

dadosFemininos = dadosFemininos.copy()
dadosFemininos['IDADE_ANOS'] = dadosFemininos['NU_IDADE_N'].apply(decode_age_years)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(dadosFemininos['IDADE_ANOS'].dropna(), bins=30, color='teal', edgecolor='white')
axes[0].set_title("Distribuição de Idade das Vítimas Femininas")
axes[0].set_xlabel("Idade (anos)")
axes[0].set_ylabel("Frequência")

# Boxplot por tipo de violência sexual
viol_sexu_label = dadosFemininos['VIOL_SEXU'].map({1.0: 'Sim', 2.0: 'Não', 9.0: 'Ignorado'})
sns.boxplot(x=viol_sexu_label, y=dadosFemininos['IDADE_ANOS'], ax=axes[1],
            palette={'Sim': 'salmon', 'Não': 'lightblue', 'Ignorado': 'lightgrey'})
axes[1].set_title("Idade × Violência Sexual")
axes[1].set_xlabel("Violência Sexual Registrada")
axes[1].set_ylabel("Idade (anos)")

plt.tight_layout()
plt.show()

print(dadosFemininos['IDADE_ANOS'].describe().round(1))


In [ ]:
# Distribuição dos tipos de violência registrados
# Valor 1 = Sim, 2 = Não/Outro, 9 = Ignorado
viol_cols = {
    'VIOL_FISIC': 'Física',
    'VIOL_PSICO': 'Psicológica',
    'VIOL_TORT':  'Tortura',
    'VIOL_SEXU':  'Sexual',
    'VIOL_TRAF':  'Tráfico',
    'VIOL_FINAN': 'Financeira',
    'VIOL_NEGLI': 'Negligência',
    'VIOL_INFAN': 'Infantil',
    'VIOL_LEGAL': 'Legal',
    'VIOL_OUTR':  'Outra',
}

prevalencia = {
    label: (dadosFemininos[col] == 1).sum()
    for col, label in viol_cols.items()
    if col in dadosFemininos.columns
}
prevalencia = dict(sorted(prevalencia.items(), key=lambda x: x[1], reverse=True))

plt.figure(figsize=(10, 5))
plt.bar(prevalencia.keys(), prevalencia.values(), color=sns.color_palette("Set2", len(prevalencia)))
plt.title("Prevalência dos Tipos de Violência — Vítimas Femininas")
plt.ylabel("Número de registros (Sim)")
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print("Percentual de Sim (valor=1) por tipo de violência:")
for col, label in viol_cols.items():
    if col in dadosFemininos.columns:
        pct = (dadosFemininos[col] == 1).mean() * 100
        print(f"  {label:20s}: {pct:5.1f} %")


In [ ]:
# Análise do perfil do agressor
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sexo do agressor
sexo_agr = dadosFemininos['AUTOR_SEXO'].map({1.0: 'Masculino', 2.0: 'Feminino',
                                              3.0: 'Ambos', 9.0: 'Ignorado'})
sexo_agr.value_counts().plot(kind='bar', ax=axes[0], color=sns.color_palette("pastel"))
axes[0].set_title("Sexo do Agressor")
axes[0].set_xlabel("")
axes[0].set_ylabel("Contagem")
axes[0].tick_params(axis='x', rotation=30)

# Relação vítima–agressor (top 8)
rel_cols = {
    'REL_CONJ': 'Cônjuge/Companheiro', 'REL_EXCON': 'Ex-cônjuge',
    'REL_NAMO': 'Namorado(a)',          'REL_EXNAM': 'Ex-namorado(a)',
    'REL_PAI':  'Pai',                  'REL_MAE':  'Mãe',
    'REL_FILHO':'Filho(a)',             'REL_DESCO':'Desconhecido',
    'REL_IRMAO':'Irmão/Irmã',          'REL_CONHEC':'Conhecido',
}
rel_counts = {
    label: (dadosFemininos[col] == 1).sum()
    for col, label in rel_cols.items()
    if col in dadosFemininos.columns
}
rel_counts = dict(sorted(rel_counts.items(), key=lambda x: x[1], reverse=True))
axes[1].barh(list(rel_counts.keys()), list(rel_counts.values()),
             color=sns.color_palette("Set3", len(rel_counts)))
axes[1].set_title("Relação Vítima–Agressor")
axes[1].set_xlabel("Contagem (Sim)")

plt.tight_layout()
plt.show()

# Uso de álcool pelo agressor
alco = dadosFemininos['AUTOR_ALCO'].map({1.0: 'Sim', 2.0: 'Não', 9.0: 'Ignorado'})
print("Uso de álcool pelo agressor:")
print(alco.value_counts(normalize=True).mul(100).round(1).to_string())


In [ ]:
# Estatísticas descritivas e ocorrência repetida
print("=== Estatísticas descritivas (numéricas) ===")
display(dadosFemininos[['IDADE_ANOS']].describe().round(2))

print()
print("=== Violência repetida (OUT_VEZES) ===")
repeticao = dadosFemininos['OUT_VEZES'].map({1.0: 'Sim — ocorreu outras vezes',
                                              2.0: 'Não — primeira vez',
                                              9.0: 'Ignorado'})
print(repeticao.value_counts(normalize=True).mul(100).round(1).to_string())

print()
print("=== Local de ocorrência (TOP 5) ===")
local_map = {1.0:'Residência', 2.0:'Habitação coletiva', 3.0:'Escola',
             4.0:'Local de prática esportiva', 5.0:'Bar/Boate',
             6.0:'Via pública', 7.0:'Comércio/Serviços', 8.0:'Indústrias',
             9.0:'Outros', 99.0:'Ignorado'}
local_lbl = dadosFemininos['LOCAL_OCOR'].map(local_map)
print(local_lbl.value_counts(normalize=True).mul(100).round(1).head(5).to_string())


---
## 3️⃣ Pré-processamento de dados
---
> Limpeza, seleção de features, tratamento de valores ausentes/ignorados,
> encoding e análise de correlação.


In [ ]:
# ── 3.1 Seleção das features relevantes para o modelo ─────────────────────────
# Target: VIOL_SEXU (1 = violência sexual registrada, 0 = demais)

FEATURES_NUMERICAS = ['IDADE_ANOS']

FEATURES_BINARIAS = [
    # Tipos de violência co-ocorrentes
    'VIOL_FISIC', 'VIOL_PSICO', 'VIOL_TORT', 'VIOL_FINAN',
    'VIOL_NEGLI', 'VIOL_INFAN', 'VIOL_LEGAL', 'VIOL_OUTR',
    # Automutilação / lesão autoprovocada
    'LES_AUTOP',
    # Perfil agressor
    'AUTOR_ALCO',
    # Repetição
    'OUT_VEZES',
]

FEATURES_CATEGORICAS = [
    'CS_GESTANT',   # gestação
    'CS_RACA',      # raça/cor
    'CS_ESCOL_N',   # escolaridade
    'LOCAL_OCOR',   # local de ocorrência
    'AUTOR_SEXO',   # sexo do agressor
]

TARGET = 'VIOL_SEXU'

all_features = FEATURES_NUMERICAS + FEATURES_BINARIAS + FEATURES_CATEGORICAS

# Colunas disponíveis no dataset
all_features = [c for c in all_features if c in dadosFemininos.columns]

df_model = dadosFemininos[all_features + [TARGET]].copy()
print(f"Shape inicial para modelagem: {df_model.shape}")
df_model.head(3)


In [ ]:
# ── 3.2 Tratamento da variável alvo ───────────────────────────────────────────
# 1 = violência sexual; 2 = não; 9 = ignorado → removemos ignorados, binarizamos
df_model = df_model[df_model[TARGET].isin([1.0, 2.0])].copy()
df_model[TARGET] = (df_model[TARGET] == 1.0).astype(int)

print("Distribuição do target (VIOL_SEXU):")
print(df_model[TARGET].value_counts(normalize=True).mul(100).round(1))
print(f"Proporção positiva: {df_model[TARGET].mean()*100:.1f} %")


In [ ]:
# ── 3.3 Encoding de variáveis binárias (1=Sim, 2=Não, 9=Ignorado) ─────────────
# Convertemos: 1→1, 2→0, 9→NaN (ignorado vira ausente)
for col in FEATURES_BINARIAS:
    if col in df_model.columns:
        df_model[col] = df_model[col].replace(9.0, np.nan)
        df_model[col] = (df_model[col] == 1.0).astype('float')

# ── 3.4 Encoding de variáveis categóricas ──────────────────────────────────────
# Substituímos 9/99 por NaN e aplicamos One-Hot Encoding
cat_replace = {9: np.nan, 9.0: np.nan, 99: np.nan, 99.0: np.nan}
for col in FEATURES_CATEGORICAS:
    if col in df_model.columns:
        df_model[col] = df_model[col].replace(cat_replace)

df_model = pd.get_dummies(df_model, columns=[c for c in FEATURES_CATEGORICAS if c in df_model.columns],
                          dummy_na=False, dtype=float)

print(f"Shape após encoding: {df_model.shape}")
print(f"Colunas: {list(df_model.columns)}")


In [ ]:
# ── 3.5 Tratamento de valores ausentes ────────────────────────────────────────
from sklearn.impute import SimpleImputer

# Separa features e target
X_raw = df_model.drop(columns=[TARGET])
y     = df_model[TARGET].values

# Imputação pela mediana (robusta a outliers)
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X_raw)
X_df = pd.DataFrame(X_imputed, columns=X_raw.columns)

print("Valores ausentes após imputação:", X_df.isnull().sum().sum())
print(f"Shape final de X: {X_df.shape}")


In [ ]:
# ── 3.6 Análise de correlação com o target ────────────────────────────────────
corr_with_target = X_df.corrwith(pd.Series(y, name=TARGET)).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Correlações negativas (top 10)
corr_with_target.head(10).plot(kind='barh', ax=axes[0], color='salmon')
axes[0].set_title("Top 10 correlações negativas com VIOL_SEXU")
axes[0].axvline(0, color='black', linewidth=0.8)

# Correlações positivas (top 10)
corr_with_target.tail(10).plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title("Top 10 correlações positivas com VIOL_SEXU")
axes[1].axvline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

# Heatmap das top 15 features mais correlacionadas
top_feats = corr_with_target.abs().nlargest(15).index.tolist()
corr_matrix = X_df[top_feats].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={"size": 8})
plt.title("Matriz de Correlação — Top 15 features mais correlacionadas com o target")
plt.tight_layout()
plt.show()


---
## 4️⃣ Modelagem
---
> Separação treino/teste e criação de três modelos de classificação:
> **Regressão Logística**, **Random Forest** e **XGBoost**.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Divisão 80/20 estratificada (mantém proporção do target)
X_train, X_test, y_train, y_test = train_test_split(
    X_df.values, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Treino : {X_train.shape[0]:,} amostras  ({y_train.mean()*100:.1f} % positivos)")
print(f"Teste  : {X_test.shape[0]:,} amostras  ({y_test.mean()*100:.1f} % positivos)")

# Normalização para a Regressão Logística
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Peso para compensar desbalanceamento de classes
neg, pos = np.bincount(y_train.astype(int))
scale_pos = neg / pos

modelos = {
    'Logistic Regression': LogisticRegression(
        max_iter=500, class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=8, class_weight='balanced',
        random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        scale_pos_weight=scale_pos, eval_metric='logloss',
        random_state=42, n_jobs=-1, verbosity=0
    ),
}

print("Modelos configurados:")
for nome in modelos:
    print(f"  ✅ {nome}")


---
## 5️⃣ Treinamento e Avaliação dos Modelos
---
> Treinamos cada modelo e avaliamos com métricas de classificação.
> Como o dataset é desbalanceado e o custo de um falso-negativo (não detectar
> violência sexual) é alto, priorizamos **Recall** e **F1-score** além da Acurácia.


In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              ConfusionMatrixDisplay)

resultados = {}

for nome, modelo in modelos.items():
    t0 = time.time()
    # Regressão Logística usa dados normalizados
    Xtr = X_train_sc if nome == 'Logistic Regression' else X_train
    Xte = X_test_sc  if nome == 'Logistic Regression' else X_test

    modelo.fit(Xtr, y_train)
    y_pred  = modelo.predict(Xte)
    y_proba = modelo.predict_proba(Xte)[:, 1]

    resultados[nome] = {
        'modelo':     modelo,
        'y_pred':     y_pred,
        'y_proba':    y_proba,
        'accuracy':   accuracy_score (y_test, y_pred),
        'precision':  precision_score(y_test, y_pred, zero_division=0),
        'recall':     recall_score   (y_test, y_pred, zero_division=0),
        'f1':         f1_score       (y_test, y_pred, zero_division=0),
        'roc_auc':    roc_auc_score  (y_test, y_proba),
        'tempo_s':    time.time() - t0,
    }
    print(f"✅ {nome} treinado em {resultados[nome]['tempo_s']:.1f}s")


In [ ]:
# ── Tabela comparativa de métricas ────────────────────────────────────────────
metricas = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
df_res = pd.DataFrame(
    {nome: {m: resultados[nome][m] for m in metricas} for nome in resultados}
).T.round(4)

df_res.columns = ['Acurácia', 'Precisão', 'Recall', 'F1-Score', 'ROC-AUC']
print("=== Comparativo de Métricas ===")
display(df_res.style.highlight_max(color='lightgreen', axis=0)
               .highlight_min(color='#ffcccc', axis=0))


In [ ]:
# ── Matrizes de confusão ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (nome, res) in zip(axes, resultados.items()):
    disp = ConfusionMatrixDisplay.from_predictions(
        y_test, res['y_pred'],
        display_labels=['Não sexual', 'Sexual'],
        ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(nome, fontsize=12)

plt.suptitle("Matrizes de Confusão — Conjunto de Teste", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc

# ── Curvas ROC ────────────────────────────────────────────────────────────────
plt.figure(figsize=(8, 6))
cores = {'Logistic Regression': 'royalblue',
         'Random Forest': 'forestgreen',
         'XGBoost': 'darkorange'}

for nome, res in resultados.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{nome} (AUC = {roc_auc:.3f})",
             color=cores[nome], lw=2)

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Classificador aleatório')
plt.xlabel("Taxa de Falso Positivo (FPR)")
plt.ylabel("Taxa de Verdadeiro Positivo (TPR / Recall)")
plt.title("Curvas ROC — Detecção de Violência Sexual")
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
# ── Feature Importance — Random Forest ───────────────────────────────────────
rf_model      = resultados['Random Forest']['modelo']
feature_names = X_df.columns.tolist()
importances   = pd.Series(rf_model.feature_importances_, index=feature_names)
top15         = importances.nlargest(15)

plt.figure(figsize=(10, 6))
top15.sort_values().plot(kind='barh', color='forestgreen')
plt.title("Feature Importance — Random Forest (Top 15)")
plt.xlabel("Importância média (Gini)")
plt.tight_layout()
plt.show()

print("Top 15 features mais importantes:")
print(top15.sort_values(ascending=False).to_string())


In [ ]:
import shap

# ── SHAP — Explicabilidade do XGBoost ─────────────────────────────────────────
xgb_model = resultados['XGBoost']['modelo']
explainer  = shap.TreeExplainer(xgb_model)

# Amostra de até 2000 registros para agilizar o cálculo
np.random.seed(42)
idx_sample = np.random.choice(len(X_test), size=min(2000, len(X_test)), replace=False)
X_sample   = X_test[idx_sample]

shap_values = explainer.shap_values(X_sample)

print("=== SHAP Summary Plot — XGBoost ===")
shap.summary_plot(shap_values, X_sample, feature_names=feature_names,
                  max_display=15, show=True, plot_type="bar")

print("\n=== SHAP Beeswarm Plot — XGBoost ===")
shap.summary_plot(shap_values, X_sample, feature_names=feature_names,
                  max_display=15, show=True)


In [ ]:
# ── Relatório de classificação detalhado (melhor modelo por F1) ───────────────
melhor = max(resultados, key=lambda n: resultados[n]['f1'])
print(f"🏆 Melhor modelo por F1-Score: {melhor}\n")
print(classification_report(y_test, resultados[melhor]['y_pred'],
                             target_names=['Não sexual', 'Sexual']))


---
## 6️⃣ Interpretação Crítica dos Resultados
---

### Métricas escolhidas e justificativa
Num sistema de triagem médica, **errar para o lado da segurança** é preferível.
Um **falso-negativo** (não identificar uma vítima de violência sexual) tem custo
muito mais alto do que um falso-positivo (acionar protocolos desnecessariamente).
Por isso priorizamos **Recall** e **F1-score** em detrimento da acurácia pura.
O **ROC-AUC** complementa a análise avaliando o modelo em todos os limiares de
decisão possíveis.

### O que os modelos aprenderam
As features com maior impacto identificadas pelo SHAP/Feature Importance revelam
que **tipo de violência co-ocorrente** (física, psicológica) e **características
do agressor** (sexo, uso de álcool) são os principais sinais preditivos, em
consonância com a literatura sobre violência doméstica.

### Pode ser usado na prática?
O modelo **pode apoiar** a triagem clínica, funcionando como:
- Um **alerta automático** nos sistemas de prontuário eletrônico quando os campos
  de notificação sugerem alta probabilidade de violência sexual;
- Uma **ferramenta de priorização** para que enfermeiras/agentes de saúde identifiquem
  casos que precisam de encaminhamento urgente (protocolo de violência sexual do
  Ministério da Saúde).

**Limitações e cuidados éticos:**
- O modelo foi treinado em notificações já **registradas**; casos não notificados
  (subnotificação) criam viés;
- Dados com muitos campos ignorados (9) reduzem a qualidade das previsões;
- O **profissional de saúde deve sempre ter a palavra final** no diagnóstico;
- Qualquer uso do modelo deve estar em conformidade com a **LGPD (Lei nº 13.709/2018)**.

### Próximos passos sugeridos
1. Coleta de dados mais completa (reduzir `9 = ignorado`);
2. Validação com dados de outros anos e outras UFs;
3. Explorar modelos interpretáveis (e.g., EBM — Explainable Boosting Machine);
4. Integração com sistemas de prontuário eletrônico (RES/RNDS).
